In [10]:
function edit_distance_uninf(x, y)
    xs = collect(x)
    ys = collect(y)
    n = length(xs)
    m = length(ys)

    D = Array{Int}(undef, n+1, m+1)

    for i in 0:n
        D[i+1, 1] = i
    end
    for j in 0:m
        D[1,j+1] = j
    end

    for i in 1:n
        for j in 1:m
            if xs[i] == ys[j]
                sub = 0
            else
                sub = 1
            end
            a = D[i, j+ 1] + 1
            b = D[i+ 1,j] + 1
            c = D[i, j] + sub
            D[i+1, j+1] = min(a, b, c)
        end
    end

    return D[n+1,m+1], D
end


x = "ispellnig"
y = "misspelling"
dist, table = edit_distance_uninf(x, y)
println(dist)


4


In [11]:
function load_kappa(path)
    dic = Dict{Tuple{Char,Char},Float64}()
    kmax = 0.0
    io = open(path, "r")
    for line in eachline(io)
        s = strip(line)
        parts = split(s)
        if length(parts) < 3
            continue
        end
        a = first(parts[1])
        b = first(parts[2])
        d = parse(Float64, parts[3])
        dic[(a,b)] = d
        dic[(b,a)] = d
        dic[(a,a)] = 0.0
        dic[(b,b)] = 0.0
        if d > kmax
            kmax = d
        end
    end
    close(io)

    if kmax == 0.0
        kmax = 1.0
    end

    K = Dict{Tuple{Char,Char},Float64}()
    for (k, d) in dic
        K[k] = d / kmax
    end
    return K
end

function edit_distance_info(x, y, K)
    xs = collect(x)
    ys = collect(y)
    n = length(xs)
    m = length(ys)

    D = Array{Float64}(undef, n + 1, m + 1)

    for i in 0:n
        D[i+1,1] = i*1.0
    end
    for j in 0:m
        D[1,j+1] = j*1.0
    end

    for i in 1:n
        for j in 1:m

            subc= K[ (uppercase(xs[i]), uppercase(ys[j]))]
            a = D[i, j + 1] + 1.0
            b = D[i + 1, j] + 1.0
            c = D[i, j] + subc
            D[i + 1, j + 1] = min(a, b, c)
        end
    end

    return D[n + 1, m + 1], D
end


K = load_kappa("keyboard_layout_distance.txt")
println(K)

x = "ispellnig"
y = "misspelling"
dist, table = edit_distance_info(x, y, K)
println(dist)



Dict(('G', 'B') => 0.12423447069116361, ('N', 'V') => 0.22222222222222224, ('I', 'B') => 0.3344998541848936, ('K', 'X') => 0.6211140274132401, ('U', 'A') => 0.6484689413823272, ('W', 'J') => 0.5938174394867308, ('Q', 'A') => 0.11455234762321377, ('Q', 'Y') => 0.5555555555555556, ('G', 'G') => 0.0, ('H', 'O') => 0.3251093613298338, ('Y', 'I') => 0.22222222222222224, ('R', 'T') => 0.11111111111111112, ('Q', 'Q') => 0.0, ('Z', 'U') => 0.6242053076698746, ('L', 'Q') => 0.9233595800524935, ('X', 'N') => 0.4444444444444445, ('G', 'E') => 0.27360746573344996, ('Y', 'K') => 0.27360746573344996, ('M', 'V') => 0.33333333333333337, ('A', 'I') => 0.7581802274715661, ('V', 'P') => 0.6242053076698746, ('V', 'M') => 0.33333333333333337, ('E', 'N') => 0.4722076407115777, ('A', 'T') => 0.431204432779236, ('X', 'G') => 0.2991542723826189, ('X', 'P') => 0.8356372120151648, ('S', 'Z') => 0.12423447069116361, ('D', 'C') => 0.12423447069116361, ('Q', 'H') => 0.5938174394867308, ('M', 'Y') => 0.2953047535724

In [13]:
using Random
using Printf

function load_words(path)
    Y = String[]
    io = open(path, "r")
    for line in eachline(io)
        w = strip(lowercase(line))
        if w != ""
            push!(Y, w)
        end
    end
    close(io)
    return Y
end

function typo_uninformed(w, p, rng)
    letters = collect('a':'z')
    cs = collect(w)
    for i in eachindex(cs)
        if rand(rng) < p
            a = cs[i]
            b = a
            while b == a
                b = letters[rand(rng, 1:length(letters))]
            end
            cs[i] = b
        end
    end
    return String(cs)
end

function typo_informed(w, p, K, rng)
    letters = collect('a':'z')
    cs = collect(w)
    beta = 8.0
    for i in eachindex(cs)
        if rand(rng) < p
            a = cs[i]
            ws = Float64[]
            bs = Char[]
            for b in letters
                if b == a
                    continue
                end
                k = K[ (uppercase(a), uppercase(b))]
                push!(bs, b)
                push!(ws, exp(-beta * k))
            end
            s = sum(ws)
            u = rand(rng) * s
            acc = 0.0
            chosen = bs[1]
            for t in 1:length(ws)
                acc += ws[t]
                if acc >= u
                    chosen = bs[t]
                    break
                end
            end
            cs[i] = chosen
        end
    end
    return String(cs)
end

function project_uninformed(x, Y)
    lx = length(x)
    bestw = ""
    bestd = Inf
    for y in Y
        if abs(length(y) - lx) > 2
            continue
        end
        d,D = edit_distance_uninf(x, y)
        if d < bestd
            bestd = d
            bestw = y
            if bestd == 0
                break
            end
        end
    end
    return bestw
end

function project_informed(x, Y, K)
    lx = length(x)
    bestw = ""
    bestd = Inf
    for y in Y
        if abs(length(y) - lx) > 2
            continue
        end
        d,D = edit_distance_info(x, y, K)
        if d < bestd
            bestd = d
            bestw = y
            if bestd == 0
                break
            end
        end
    end
    return bestw
end

function run_q3(words_path, keyboard_path)
    rng = MersenneTwister(1)
    Yall = load_words(words_path)
    K = load_kappa(keyboard_path)

    Y = Yall
    if length(Yall) > 20000
        idx = rand(rng, 1:length(Yall), 20000)
        Y = Yall[idx]
    end

    N = 300
    T = Y
    if length(Y) > N
        idx = rand(rng, 1:length(Y), N)
        T = Y[idx]
    end

    ps = [0.00, 0.02, 0.05, 0.08, 0.10, 0.12]

    @printf("%8s | %10s | %10s\n", "p", "uninformed", "informed")
    println("----------+------------+------------")

    for p in ps
        ok_u = 0
        ok_i = 0

        for w in T
            xu = typo_uninformed(w, p, rng)
            xi = typo_informed(w, p, K, rng)

            pu = project_uninformed(xu, Y)
            pi = project_informed(xi, Y, K)

            if pu == w
                ok_u += 1
            end
            if pi == w
                ok_i += 1
            end
        end

        sr_u = ok_u / length(T)
        sr_i = ok_i / length(T)
        @printf("%8.2f | %10.3f | %10.3f\n", p, sr_u, sr_i)
    end
end

run_q3("words_alpha.txt", "keyboard_layout_distance.txt")


       p | uninformed |   informed
----------+------------+------------
    0.00 |      1.000 |      1.000
    0.02 |      0.997 |      0.997
    0.05 |      0.987 |      0.993
    0.08 |      0.987 |      0.997
    0.10 |      0.980 |      0.993
    0.12 |      0.960 |      0.977
